In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-05-20 08:31:58.494895: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747729918.918230      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747729919.039224      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
test_message = "Emergency B+ve blood needed at Rampura. Contact: 01777123432"

In [4]:
lora_model_name = "cbrs-parsing-lora-v1-llama-3b"

In [5]:
from datasets import load_dataset

dataset = load_dataset("imAniksahA/CBRS-parsing", split={
    'train': 'train',
    'validation': 'validation',
    'test': 'test'
})

train.jsonl:   0%|          | 0.00/8.87M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7865 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/983 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/984 [00:00<?, ? examples/s]

In [6]:
dataset['test'][0]["conversations"]

[{'from': 'human',
  'value': '#Guntur AP 2 Units O-ve #Blood #urgent #need for #StomachOperation at Omega Hospital Auto Nagar Pls Call 9490761930 or 9848160312 #July_10 via @IamKalyanRaksha cc @trulymaheshh #BloodMatters'},
 {'from': 'gpt',
  'value': '{"blood_group": "O-", "bags_needed": "2", "patient": {"name": "", "gender": "", "age_group": ""}, "condition": "stomach operation", "location": "Omega Hospital Auto Nagar", "hospital_name": "Omega Hospital Auto Nagar", "location_markers": ["Guntur"], "probable_day": "10/07", "probable_time": "", "contacts": [{"name": "", "contact_numbers": ["9490761930", "9848160312"], "relation_with_patient": ""}], "compensation": {"transportation": "", "allowance": ""}}'}]

In [7]:
data = []

for d in dataset['test']:
    convo = d['conversations']
    data.append({
        'text': convo[0]['value'],
        'ref_json': convo[1]['value']
    })
data[:3]

[{'text': '#Guntur AP 2 Units O-ve #Blood #urgent #need for #StomachOperation at Omega Hospital Auto Nagar Pls Call 9490761930 or 9848160312 #July_10 via @IamKalyanRaksha cc @trulymaheshh #BloodMatters',
  'ref_json': '{"blood_group": "O-", "bags_needed": "2", "patient": {"name": "", "gender": "", "age_group": ""}, "condition": "stomach operation", "location": "Omega Hospital Auto Nagar", "hospital_name": "Omega Hospital Auto Nagar", "location_markers": ["Guntur"], "probable_day": "10/07", "probable_time": "", "contacts": [{"name": "", "contact_numbers": ["9490761930", "9848160312"], "relation_with_patient": ""}], "compensation": {"transportation": "", "allowance": ""}}'},
 {'text': 'একজন মুমূর্ষু রোগীর জন্য জরুরি রক্ত প্রয়োজন \n🔴রক্তের গ্রুপ:O+\n🌡রক্তের পরিমাণ:2 ব্যাগ \n🏨রক্তদানের স্থান: রয়েল হসপিটাল,কুমিল্লা \n📱যোগাযোগ:01763855487',
  'ref_json': '{"blood_group": "O+", "bags_needed": "2", "patient": {"name": "", "gender": "", "age_group": ""}, "condition": "", "location": "\\u09b0\

In [8]:
# Import necessary libraries
from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer

# Define parameters
# lora_model_name = "imAniksahA/cbrs-parsing-lora-v1-llama-3b"
lora_model_name = "imAniksahA/cbrs-parsing-lora-v3-epoch-3-llama-3b"
max_seq_length = 2048  # Adjust based on your needs
dtype = torch.float16  # Use float16 for efficiency on Kaggle
load_in_4bit = True    # Use 4-bit quantization for memory efficiency

# Load the model and tokenizer
if True:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=lora_model_name,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    FastLanguageModel.for_inference(model)  # Enable faster inference

test_message = "Emergency B+ve blood needed at Rampura. Contact: 01777123432"

# Create chat messages
messages = [
    {"role": "user", "content": test_message},
]

# Tokenize the input
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Required for generation
    return_tensors="pt",
).to("cuda")

# Initialize text streamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# Generate response
_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=4096,
    use_cache=True,
    temperature=1.5,
    min_p=0.1
)

==((====))==  Unsloth 2025.5.6: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/195M [00:00<?, ?B/s]

Unsloth 2025.5.6 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


{"blood_group": "B+", "bags_needed": "", "patient": {"name": "", "gender": "", "age_group": ""}, "condition": "", "location": "Rampura", "hospital_name": "", "location_markers": ["Rampura"], "probable_day": "", "probable_time": "", "contacts": [{"name": "", "contact_numbers": ["01777123432"], "relation_with_patient": ""}], "compensation": {"transportation": "", "allowance": ""}}<|eot_id|>


In [9]:
# import torch
# from unsloth import FastLanguageModel

# # Define an array of test messages
# test_messages = [
#     "Emergency B+ve blood needed at Rampura. Contact: 01777123432"
#     "Urgent AB+ve blood needed at Rampura. Transport cost will be provided Contact: 01777123432"
#     "Emergency O negative blood needed at Rampura. Contact: 017767723432, patient is Siam, contact number is of father"
#     "Emergency B positive blood needed at Sylhet Medical College and Hospital. Contact: 01774423432"
# ]

# # Prepare batched inputs
# batch_messages = [
#     [{"role": "user", "content": message}] for message in test_messages
# ]

# # Tokenize the batch of messages
# tokenized_inputs = [
#     tokenizer.apply_chat_template(
#         msg,
#         tokenize=True,
#         add_generation_prompt=True,
#         return_tensors="pt"
#     ) for msg in batch_messages
# ]

# # Pad the tokenized inputs to create a batch
# input_ids = torch.nn.utils.rnn.pad_sequence(
#     [input.squeeze(0) for input in tokenized_inputs],
#     batch_first=True,
#     padding_value=tokenizer.pad_token_id
# ).to("cuda")

# # Create attention mask for the padded inputs
# attention_mask = (input_ids != tokenizer.pad_token_id).long()

# # Perform batched inference
# outputs = model.generate(
#     input_ids=input_ids,
#     attention_mask=attention_mask,
#     max_new_tokens=128,
#     use_cache=True,
#     temperature=1.5,
#     min_p=0.1,
#     pad_token_id=tokenizer.pad_token_id,
#     eos_token_id=tokenizer.eos_token_id,
#     return_dict_in_generate=True,
#     output_scores=False
# )

# # Decode the outputs into an array
# generated_texts = []
# for output in outputs.sequences:
#     # Decode the generated tokens, skipping special tokens
#     generated_text = tokenizer.decode(output, skip_special_tokens=True)
#     # Extract the generated part (after the prompt)
#     prompt_length = len(tokenizer.decode(input_ids[0], skip_special_tokens=True))  # Assuming uniform prompt length
#     generated_only = generated_text[prompt_length:].strip()
#     generated_texts.append(generated_only)

# # Print the resulting array of generated texts
# print("Generated texts:", generated_texts)

In [10]:
# data = data[:1]
# data

In [11]:
import json

In [12]:
# Process each entry and generate parsed_json
results = []
for entry in data:
    # Prepare the input message
    messages = [
        {"role": "user", "content": entry['text']}
    ]

    # Tokenize the input
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate response without streaming
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=4096,
        use_cache=True,
        temperature=1.5,
        min_p=0.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=False
    )

    # Decode the generated output
    generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    # Extract the parsed_json (assuming the model outputs a JSON-like string after the prompt)
    prompt_length = len(tokenizer.decode(inputs[0], skip_special_tokens=True))
    parsed_json = generated_text[prompt_length:].strip()

    # Append the result
    results.append({
        'text': entry['text'],
        'ref_json': entry['ref_json'],
        'parsed_json': parsed_json
    })

# Save the results to a JSON file
with open('parsed_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("Results saved to 'parsed_results.json'")

Results saved to 'parsed_results.json'
